In [1]:

!pip -q install scikit-learn

import os, shutil, itertools, datetime, json, math, time
import numpy as np, matplotlib.pyplot as plt
from pathlib import Path
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import EfficientNetB3
from tensorflow.keras.applications.efficientnet import preprocess_input

In [2]:
# 1) Mount Drive (if not yet)
# -------------------------
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# ---------------- Mount & Paths ----------------
drive.mount('/content/drive')

DRIVE_ROOT = "/content/drive/MyDrive/Colab Notebooks"
DATA_DIR  = f"{DRIVE_ROOT}/emotion/emotion_preprocessed_split70_30_aug4000"  # train & val
TEST_DIR  = f"{DRIVE_ROOT}/emotion/emotion_preprocessed/test"

LOCAL_TRAIN = "/content/data/train"
LOCAL_VAL   = "/content/data/val"
LOCAL_TEST  = "/content/data/test"

ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
OUT_DIR = f"{DRIVE_ROOT}/DL/Emotion_EfficientNetB3_Results_{ts}"
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)

def fresh_copy(src, dst):
    if os.path.exists(dst): shutil.rmtree(dst)
    shutil.copytree(src, dst)

fresh_copy(os.path.join(DATA_DIR, "train"), LOCAL_TRAIN)
fresh_copy(os.path.join(DATA_DIR, "val"),   LOCAL_VAL)
fresh_copy(TEST_DIR, LOCAL_TEST)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
# Reproducibility + speed
SEED         = 42
TRAIN_BATCH  = 32      # VRAM টাইট হলে 16
TEST_BATCH   = 256     # predict দ্রুত; ধীর হলে 128
IMG_SIZE_RAW = (48, 48)
IMG_SIZE_EFF = (300, 300)
USE_AUG      = False   # ← চাইলে True করো

tf.keras.utils.set_random_seed(SEED)
try:
    tf.keras.mixed_precision.set_global_policy('mixed_float16')
    tf.config.optimizer.set_jit(True)
    print("✅ Mixed precision + XLA enabled")
except Exception as e:
    print("⚠️ Mixed precision/XLA not enabled:", e)

✅ Mixed precision + XLA enabled


In [5]:
# 3) tf.data loaders (48×48 gray → RGB → 300×300 → preprocess_input)
# -------------------------
AUTOTUNE = tf.data.AUTOTUNE

augment = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.1),
    layers.RandomTranslation(0.05, 0.05)
], name="aug")

def make_train_loader(dir_path, shuffle=True, batch=TRAIN_BATCH):
    ds = tf.keras.utils.image_dataset_from_directory(
        dir_path, labels="inferred", label_mode="categorical",
        color_mode="grayscale", image_size=IMG_SIZE_RAW,
        batch_size=batch, shuffle=shuffle, seed=SEED
    )
    class_names = ds.class_names

    def _prep(x, y):
        x = tf.image.grayscale_to_rgb(x)
        if USE_AUG:
            x = augment(x)
        x = tf.image.resize(x, IMG_SIZE_EFF)
        x = preprocess_input(x)
        return x, y

    ds = ds.map(_prep, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
    return ds, class_names

def make_loader_with_forced_classes(dir_path, forced_class_names, shuffle, batch=TRAIN_BATCH):
    """Val/Test loader — train-এর class_names force করা, যাতে missing class থাকলেও shape একই থাকে"""
    ds = tf.keras.utils.image_dataset_from_directory(
        dir_path, labels="inferred", label_mode="categorical",
        color_mode="grayscale", image_size=IMG_SIZE_RAW,
        batch_size=batch, shuffle=shuffle, seed=SEED,
        class_names=forced_class_names
    )
    def _prep(x, y):
        x = tf.image.grayscale_to_rgb(x)
        x = tf.image.resize(x, IMG_SIZE_EFF)
        x = preprocess_input(x)
        return x, y
    ds = ds.map(_prep, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
    return ds

# Build datasets
train_ds, class_names = make_train_loader(LOCAL_TRAIN, shuffle=True)
val_ds   = make_loader_with_forced_classes(LOCAL_VAL,  class_names, shuffle=False)
test_ds  = make_loader_with_forced_classes(TEST_DIR,   class_names, shuffle=False)
num_classes = len(class_names)
print("Classes:", class_names)

Found 28000 files belonging to 7 classes.
Found 8616 files belonging to 7 classes.
Found 7178 files belonging to 7 classes.
Classes: ['angry', 'disgusted', 'fearful', 'happy', 'neutral', 'sad', 'surprised']


In [6]:
# 4) Class weights (optional, handles imbalance)
# -------------------------
class_names_fs = sorted(tf.io.gfile.listdir(LOCAL_TRAIN))
def _count_in(folder):
    return {c: len(tf.io.gfile.listdir(os.path.join(folder, c)))
            for c in class_names_fs if tf.io.gfile.isdir(os.path.join(folder, c))}
train_counts = _count_in(LOCAL_TRAIN)
counts = np.array([train_counts.get(c, 0) for c in class_names_fs], dtype=np.float32)
total = counts.sum()
class_weight = {i: float(total / (len(class_names_fs) * c)) for i, c in enumerate(counts)}
print("Train counts:", train_counts)
print("Class weight:", class_weight)


Train counts: {'angry': 4000, 'disgusted': 4000, 'fearful': 4000, 'happy': 4000, 'neutral': 4000, 'sad': 4000, 'surprised': 4000}
Class weight: {0: 1.0, 1: 1.0, 2: 1.0, 3: 1.0, 4: 1.0, 5: 1.0, 6: 1.0}


In [7]:
# 5) Model builder + unfreeze helper
# -------------------------
def build_model(dropout=0.5):
    base = EfficientNetB3(include_top=False, weights="imagenet", input_shape=(300,300,3))
    for layer in base.layers:
        layer.trainable = False  # Phase-1: freeze
    inp = layers.Input(shape=(300,300,3))
    x = base(inp, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(512, activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(dropout)(x)
    out = layers.Dense(num_classes, activation="softmax", dtype="float32")(x)  # ensure fp32 out
    model = models.Model(inp, out)
    return model, base

def unfreeze_last_K(base, K=70, freeze_bn=True):
    for i, layer in enumerate(base.layers):
        if freeze_bn and isinstance(layer, tf.keras.layers.BatchNormalization):
            layer.trainable = False
        else:
            layer.trainable = (i >= len(base.layers) - K)


In [8]:
# 6) Train helpers (Head + Fine-tune)
# -------------------------
def train_head(model, epochs=12):
    model.compile(
        optimizer=tf.keras.optimizers.Adam(3e-4),
        loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.05),
        metrics=["accuracy"]
    )
    cbs = [
        tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, verbose=1),
        tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=7, restore_best_weights=True, verbose=1)
    ]
    hist = model.fit(
        train_ds, validation_data=val_ds,
        epochs=epochs, callbacks=cbs, class_weight=class_weight, verbose=1
    )
    return hist

def train_ft(model, base, K=70, lr_ft=1e-5, epochs=15):
    unfreeze_last_K(base, K=K, freeze_bn=True)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(lr_ft, clipnorm=1.0),
        loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.05),
        metrics=["accuracy"]
    )
    cbs = [
        tf.keras.callbacks.ModelCheckpoint(
            filepath=f"{OUT_DIR}/trial_best.keras",
            monitor="val_accuracy", mode="max", save_best_only=True, verbose=1
        ),
        tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, verbose=1, min_lr=5e-7),
        tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True, verbose=1, min_delta=1e-3)
    ]
    hist = model.fit(
        train_ds, validation_data=val_ds,
        epochs=epochs, callbacks=cbs, class_weight=class_weight, verbose=1
    )
    best_val_acc = max(hist.history["val_accuracy"])
    return hist, best_val_acc


In [9]:
# 7) Light Parameter Tuning (Turbo): head once, then try K in [60, 80]
# -------------------------
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
TARGET_ACC = 0.60          # হিট হলে টিউনিং আগেই থামবে
FT_K_LIST  = [60, 80]
LR_FT      = 1e-5
DROPOUT    = 0.5

results = []
best = {"val_acc": -1.0, "K": None, "weights": None}
print("\n🚀 Head train (one-time)")
model, base = build_model(dropout=DROPOUT)
history_head = train_head(model, epochs=12)

_head_tmp = "/content/head_trained.weights.h5"
model.save_weights(_head_tmp)

best_hist_ft = None
for i, K in enumerate(FT_K_LIST, start=1):
    print(f"\n🔎 FT Trial {i}/{len(FT_K_LIST)} | K={K}")
    t0 = time.time()
    model, base = build_model(dropout=DROPOUT)
    model.load_weights(_head_tmp)

    hist_ft, val_acc = train_ft(model, base, K=K, lr_ft=LR_FT, epochs=15)
    dt = time.time() - t0
    print(f"   → val_accuracy: {val_acc:.4f} | time: {dt/60:.1f} min")
    results.append({"K": K, "val_acc": float(val_acc), "minutes": round(dt/60, 2)})

    if val_acc > best["val_acc"]:
        best["val_acc"] = val_acc
        best["K"]       = K
        tmp_path = "/content/best_tmp.weights.h5"
        model.save_weights(tmp_path)
        with open(tmp_path, "rb") as f:
            best["weights"] = f.read()
        best_hist_ft = hist_ft

    if val_acc >= TARGET_ACC:
        print(f"✅ Target met (>= {TARGET_ACC:.2f}). Early stop tuning.")
        break

with open(f"{OUT_DIR}/tuning_results.json", "w") as f:
    json.dump({"trials": results, "best": {"K": best["K"], "val_acc": best["val_acc"]}}, f, indent=2)

print("\n🏁 Tuning done. Best K =", best["K"], "| val_acc =", round(best["val_acc"], 4))


🚀 Head train (one-time)
43941136/43941136 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/12
875/875 ━━━━━━━━━━━━━━━━━━━━ 240s 175ms/step - accuracy: 0.3471 - loss: 2.2299 - val_accuracy: 0.4965 - val_loss: 1.4496 - learning_rate: 3.0000e-04
Epoch 2/12
875/875 ━━━━━━━━━━━━━━━━━━━━ 120s 89ms/step - accuracy: 0.4688 - loss: 1.5708 - val_accuracy: 0.5243 - val_loss: 1.3544 - learning_rate: 3.0000e-04
Epoch 3/12
875/875 ━━━━━━━━━━━━━━━━━━━━ 77s 88ms/step - accuracy: 0.5103 - loss: 1.4068 - val_accuracy: 0.5353 - val_loss: 1.3404 - learning_rate: 3.0000e-04
Epoch 4/12
875/875 ━━━━━━━━━━━━━━━━━━━━ 80s 91ms/step - accuracy: 0.5447 - loss: 1.3247 - val_accuracy: 0.5414 - val_loss: 1.3151 - learning_rate: 3.0000e-04
Epoch 5/12
875/875 ━━━━━━━━━━━━━━━━━━━━ 77s 88ms/step - accuracy: 0.5633 - loss: 1.2828 - val_accuracy: 0.5448 - val_loss: 1.3060 - learning_rate: 3.0000e-04
Epoch 6/12
875/875 ━━━━━━━━━━━━━━━━━━━━ 75s 86ms/step - accuracy: 0.5718 - loss: 1.2542 - val_accuracy: 0.5521 - val_loss: 1.2992 -

In [10]:
# 8) Rebuild best model & save
# -------------------------
best_model, best_base = build_model(dropout=DROPOUT)
tmp_path = "/content/best_tmp.weights.h5"
with open(tmp_path, "wb") as f:
    f.write(best["weights"])
best_model.load_weights(tmp_path)
unfreeze_last_K(best_base, K=best["K"], freeze_bn=True)
best_model.compile(
    optimizer=tf.keras.optimizers.Adam(LR_FT),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.05),
    metrics=["accuracy"]
)
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
best_model.save(f"{OUT_DIR}/efficientnetb3_best_model.h5")
print("✅ Best model saved to:", f"{OUT_DIR}/efficientnetb3_best_model.h5")

✅ Best model saved to: /content/drive/MyDrive/Colab Notebooks/DL/Emotion_EfficientNetB3_Results_20251011_033554/efficientnetb3_best_model.h5


In [11]:
# 9) Curves (head + best FT)
# -------------------------
def plot_curves(h1, h2, outdir=OUT_DIR):
    acc      = h1.history["accuracy"]     + h2.history["accuracy"]
    val_acc  = h1.history["val_accuracy"] + h2.history["val_accuracy"]
    loss     = h1.history["loss"]         + h2.history["loss"]
    val_loss = h1.history["val_loss"]     + h2.history["val_loss"]
    epochs = range(1, len(acc)+1)
    plt.figure(figsize=(12,5))
    plt.subplot(1,2,1); plt.plot(epochs, acc, label="train"); plt.plot(epochs, val_acc, label="val"); plt.title("Accuracy"); plt.xlabel("Epoch"); plt.legend()
    plt.subplot(1,2,2); plt.plot(epochs, loss, label="train"); plt.plot(epochs, val_loss, label="val"); plt.title("Loss"); plt.xlabel("Epoch"); plt.legend()
    plt.tight_layout()
    fpath = f"{outdir}/training_curves.png"
    plt.savefig(fpath, dpi=150); plt.close()
    print("Saved:", fpath)

plot_curves(history_head, best_hist_ft, OUT_DIR)


Saved: /content/drive/MyDrive/Colab Notebooks/DL/Emotion_EfficientNetB3_Results_20251011_033554/training_curves.png


In [12]:
# 10) Evaluate on TEST (deterministic steps; no "Unknown")
# -------------------------
num_test = sum(len(tf.io.gfile.listdir(os.path.join(TEST_DIR, c)))
               for c in sorted(tf.io.gfile.listdir(TEST_DIR)))
print("num_test =", num_test)

# labels first (fresh iterator)
ds_labels = test_ds.unbatch().batch(TEST_BATCH).prefetch(1)
y_true = np.concatenate([y.numpy() for _, y in ds_labels], axis=0)
y_true_cls = np.argmax(y_true, axis=1)

# predict with known steps (fresh iterator)
ds_pred = test_ds.unbatch().batch(TEST_BATCH).prefetch(1)
steps = math.ceil(num_test / TEST_BATCH)
y_prob = best_model.predict(ds_pred, steps=steps, verbose=1)

y_pred = np.argmax(y_prob, axis=1)


num_test = 7178
29/29 ━━━━━━━━━━━━━━━━━━━━ 162s 3s/step


In [13]:
# 11) Classification report
# -------------------------
report = classification_report(y_true_cls, y_pred, target_names=class_names, digits=4)
with open(f"{OUT_DIR}/classification_report.txt", "w") as f:
    f.write(report)
print(report)
print("Saved:", f"{OUT_DIR}/classification_report.txt")


              precision    recall  f1-score   support

       angry     0.5298    0.5282    0.5290       958
   disgusted     0.6364    0.6306    0.6335       111
     fearful     0.4768    0.4209    0.4471      1024
       happy     0.8109    0.8292    0.8200      1774
     neutral     0.5688    0.6067    0.5871      1233
         sad     0.5000    0.4731    0.4862      1247
   surprised     0.7067    0.7653    0.7348       831

    accuracy                         0.6202      7178
   macro avg     0.6042    0.6077    0.6054      7178
weighted avg     0.6154    0.6202    0.6172      7178

Saved: /content/drive/MyDrive/Colab Notebooks/DL/Emotion_EfficientNetB3_Results_20251011_033554/classification_report.txt


In [14]:
# 12) Confusion matrix
# -------------------------
cm = confusion_matrix(y_true_cls, y_pred)
plt.figure(figsize=(7.2,6.2))
plt.imshow(cm, interpolation="nearest", cmap="Blues")
plt.title("Confusion Matrix"); plt.colorbar()
tick_marks = np.arange(num_classes)
plt.xticks(tick_marks, class_names, rotation=45, ha="right")
plt.yticks(tick_marks, class_names)
th = cm.max()/2.0
for I, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
    plt.text(j, I, cm[I, j], ha="center",
             color="white" if cm[I, j] > th else "black", fontsize=8)
plt.ylabel("True"); plt.xlabel("Predicted"); plt.tight_layout()
cm_path = f"{OUT_DIR}/confusion_matrix.png"
plt.savefig(cm_path, dpi=150); plt.close()
print("Saved:", cm_path)

# -------------------------


Saved: /content/drive/MyDrive/Colab Notebooks/DL/Emotion_EfficientNetB3_Results_20251011_033554/confusion_matrix.png


In [15]:
# 13) ROC (OvR + micro)
# -------------------------
fpr = {}; tpr = {}; roc_auc = {}
for i in range(num_classes):
    fpr[i], tpr[i], _ = roc_curve(y_true[:, i], y_prob[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

fpr["micro"], tpr["micro"], _ = roc_curve(y_true.ravel(), y_prob.ravel())
roc_auc["micro"] = auc(fpr["micro"], tpr["micro"])

plt.figure(figsize=(8,6))
plt.plot(fpr["micro"], tpr["micro"], label=f"micro-average ROC (AUC = {roc_auc['micro']:.3f})", linewidth=2)
for i, cname in enumerate(class_names):
    plt.plot(fpr[i], tpr[i], lw=1, label=f"{cname} (AUC = {roc_auc[i]:.3f})")
plt.plot([0,1],[0,1],"k--", lw=1)
plt.xlim([0.0,1.0]); plt.ylim([0.0,1.05])
plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
plt.title("Multi-class ROC (OvR)")
plt.legend(loc="lower right", fontsize=8)
roc_path = f"{OUT_DIR}/roc_curves.png"
plt.savefig(roc_path, dpi=150); plt.close()
print("Saved:", roc_path)

print("\n✅ All artifacts saved in:", OUT_DIR)

Saved: /content/drive/MyDrive/Colab Notebooks/DL/Emotion_EfficientNetB3_Results_20251011_033554/roc_curves.png

✅ All artifacts saved in: /content/drive/MyDrive/Colab Notebooks/DL/Emotion_EfficientNetB3_Results_20251011_033554
